# SupplySight AI

## Enterprise Procurement Decision Intelligence Platform

# Notebook 01F

# Country Weather ETL

---

## Objective

The objective of this notebook is to construct an enterprise-grade historical
Country Weather Dimension for the SupplySight AI Procurement Decision
Intelligence Platform.

Weather conditions influence procurement operations through their impact on
supplier performance, transportation efficiency, logistics disruptions and
overall supply chain risk. Integrating historical weather data enables external
environmental factors to be analyzed alongside internal procurement
transactions.

Rather than collecting weather observations for thousands of individual ERP
cities, this notebook adopts a standardized country-level enrichment strategy.
Each ERP country is represented by a single weather reference city defined in
the Enterprise Country Dimension (Notebook 01E). This approach provides
consistent geographic coverage, simplifies data integration and ensures a
scalable ETL process suitable for enterprise analytics.

Historical weather observations are retrieved from the Open-Meteo Archive API
for the period **1 January 2015** to **31 December 2018**.

The workflow includes:

- Loading the Enterprise Country Dimension
- Preparing standardized weather request locations
- Building the historical extraction calendar
- Generating the Country-Date weather request matrix
- Retrieving geographic coordinates for weather reference cities
- Applying manual geocoding corrections where required
- Downloading historical daily weather observations
- Validating weather data quality and completeness
- Exporting the Enterprise Weather Dimension

The resulting dataset will later be integrated with procurement transactions
using standardized country and date keys within the Enterprise Data Integration
Pipeline (Notebook 04A).

---

## Expected Output

This notebook generates two enterprise reference datasets:

### 1. Weather Coordinates

A reusable geographic reference table containing one standardized weather
location for every ERP country.

### 2. Enterprise Weather Dimension

A historical daily weather dataset containing:

- Country
- ISO Country Code
- Weather Reference City
- Observation Date
- Mean Daily Temperature
- Daily Precipitation
- Maximum Wind Speed

These datasets provide the external environmental context required for
procurement analytics, forecasting models and enterprise decision
intelligence.

---


## Import Libraries

In [42]:
# ============================================================
# Import Libraries
# ============================================================

import pandas as pd
import requests
import time
from pathlib import Path

## Configuration

In [43]:
# ============================================================
# Configuration
# ============================================================

PROJECT_PATH = Path(
    r"C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform"
)

PROCESSED = PROJECT_PATH / "Datasets" / "Processed Datasets"

RAW = PROJECT_PATH / "Datasets" / "Raw Datasets"

EXTERNAL = PROJECT_PATH / "Datasets" / "External Datasets"

print("Project Paths Configured Successfully")

Project Paths Configured Successfully



---

## Load Enterprise Country Dimension

The Country Dimension created in Notebook 01E serves as the master geographic
reference for external enrichment datasets.

Rather than identifying locations directly from ERP transactions, all
geographic information is standardized through this dimension.

The weather reference city assigned to each country will be used to retrieve
historical weather observations.

In [4]:
# ============================================================
# Load Country Dimension
# ============================================================

dim_country = pd.read_csv(
    PROCESSED / "dim_country.csv",
    dtype={"iso2": str},
    keep_default_na=False
)

print("="*70)
print("COUNTRY DIMENSION")
print("="*70)

print(f"Rows    : {dim_country.shape[0]}")
print(f"Columns : {dim_country.shape[1]}")

display(dim_country.head())

COUNTRY DIMENSION
Rows    : 164
Columns : 14


,country_key,erp_country,country_name,iso2,iso3,region,subregion,continent,capital,weather_reference_city,languages,geoname_id,created_at,data_source
0,1,Afganistán,Afghanistan,AF,AFG,Asia,Southern Asia,AS,Kabul,Kabul,"fa-AF,ps,uz-AF,tk",1149361.0,2026-07-15 16:55:19.515053,DataHub Country Codes
1,2,Albania,Albania,AL,ALB,Europe,Southern Europe,EU,Tirana,Tirana,"sq,el",783754.0,2026-07-15 16:55:19.515053,DataHub Country Codes
2,3,Alemania,Germany,DE,DEU,Europe,Western Europe,EU,Berlin,Berlin,de,2921044.0,2026-07-15 16:55:19.515053,DataHub Country Codes
3,4,Angola,Angola,AO,AGO,Africa,Sub-Saharan Africa,AF,Luanda,Luanda,pt-AO,3351879.0,2026-07-15 16:55:19.515053,DataHub Country Codes
4,5,Arabia Saudí,Saudi Arabia,SA,SAU,Asia,Western Asia,AS,Riyadh,Riyadh,ar-SA,102358.0,2026-07-15 16:55:19.515053,DataHub Country Codes



---

### 🔍 Overview

- The **Country Dimension (`dim_country`)** acts as the **single source of truth** for all geographic data.
- It standardizes country-related attributes across datasets instead of relying on ERP transaction values.
- The column **`weather_reference_city`** is critical for linking external weather datasets.

### 📊 Key Observations

- **Dataset Size**
  - 164 rows (countries)
  - 14 columns (rich geographic metadata)

- **Data Completeness**
  - Core geographic fields are populated:
    - `country_name`, `iso2`, `iso3`, `region`, `continent`
  - `weather_reference_city` is available for each country → enables weather enrichment

- **Standardization Benefits**
  - Eliminates inconsistencies from ERP country names (e.g., language variations like *Alemania → Germany*)
  - Provides normalized ISO codes for integration with external APIs

- **Multilingual Support**
  - `languages` column contains multiple language codes per country
  - Useful for localization but not directly needed for weather joins

- **Geospatial Reference**
  - `geoname_id` provides a link to external geographic datasets (e.g., GeoNames)

### ⚠️ Potential Issues

- **City-Level Approximation**
  - Weather is based on a **single reference city per country**
  - May not represent:
    - Large countries (e.g., USA, India, Brazil)
    - Countries with diverse climates

- **Missing or Inconsistent IDs**
  - `geoname_id` may contain nulls or floats (should ideally be integer)

- **ERP vs Standard Names**
  - `erp_country` may differ from `country_name`
  - Requires careful mapping during joins

### ✅ Final Outcome

- A **clean, standardized country dimension** ready for:
  - Weather data enrichment 🌦️
  - Cross-dataset joins 🔗
  - Consistent geographic reporting 📍

- Enables building a **robust external data pipeline** aligned with enterprise data modeling best practices.

---



## Prepare Weather Request Table

Only the attributes required for weather retrieval are selected from the
Country Dimension.

This request table provides one standardized weather location for every ERP
country and forms the basis for historical weather extraction.

In [5]:
# ============================================================
# Prepare Weather Request Table
# ============================================================

weather_requests = dim_country[
    [
        "iso2",
        "country_name",
        "weather_reference_city"
    ]
].copy()

weather_requests.rename(
    columns={
        "country_name":"country",
        "weather_reference_city":"city"
    },
    inplace=True
)

print("="*70)
print("WEATHER REQUEST TABLE")
print("="*70)

print(f"Countries : {len(weather_requests)}")

display(weather_requests.head())

WEATHER REQUEST TABLE
Countries : 164


,iso2,country,city
0,AF,Afghanistan,Kabul
1,AL,Albania,Tirana
2,DE,Germany,Berlin
3,AO,Angola,Luanda
4,SA,Saudi Arabia,Riyadh



---

### 🔍 Overview

- The **Weather Request Table (`weather_requests`)** is derived from the Country Dimension.
- It contains only the **essential attributes required for weather API calls**:
  - `iso2` → country code
  - `country` → standardized country name
  - `city` → weather reference city
- Each row represents **one standardized weather location per country**.

### 📊 Key Observations

- **Dataset Size**
  - 164 countries → matches the Country Dimension
  - Ensures **full geographic coverage**

- **Simplified Structure**
  - Reduced from 14 columns → 3 columns
  - Optimized for API usage and external data integration

- **Standardization**
  - Column renaming improves clarity:
    - `country_name` → `country`
    - `weather_reference_city` → `city`
  - ISO2 codes ensure compatibility with most weather APIs

- **One-to-One Mapping**
  - Each country has exactly **one weather reference city**
  - Prevents duplication and simplifies joins

### ⚠️ Potential Issues

- **Single City Limitation**
  - Weather data may not represent entire countries:
    - Large countries (e.g., India, USA) have multiple climates
  - May introduce bias in analytics

- **City Name Ambiguity**
  - Some city names may not be unique globally
  - APIs may require:
    - Country code (`iso2`)
    - Or coordinates (lat/long)

- **Data Quality Dependency**
  - Accuracy depends entirely on:
    - `weather_reference_city` correctness
    - Upstream Country Dimension quality

### ✅ Final Outcome

- A **lean and standardized request table** ready for:
  - Historical weather data extraction 🌦️
  - API integration 🔗
  - Scalable ETL pipelines ⚙️

- Acts as a **bridge between enterprise data and external weather services**, ensuring consistency and reliability.

---


## Build Historical Calendar

A complete daily calendar is generated for the historical analysis period.

Every country will later be combined with every calendar date to create the
complete set of weather requests.

This ensures consistent temporal coverage across all countries.

In [6]:
# ============================================================
# Build Historical Calendar
# ============================================================

calendar = pd.DataFrame({

    "date": pd.date_range(

        start="2015-01-01",
        end="2018-12-31",
        freq="D"

    )

})

print("="*70)
print("HISTORICAL CALENDAR")
print("="*70)

print(f"Days : {len(calendar)}")

display(calendar.head())
display(calendar.tail())

HISTORICAL CALENDAR
Days : 1461


,date
0,2015-01-01
1,2015-01-02
2,2015-01-03
3,2015-01-04
4,2015-01-05


,date
1456,2018-12-27
1457,2018-12-28
1458,2018-12-29
1459,2018-12-30
1460,2018-12-31



---

### 🔍 Overview

- A **daily calendar (`calendar`)** is generated covering the full analysis period:
  - **Start Date:** 2015-01-01  
  - **End Date:** 2018-12-31  
- This calendar will be used to **combine with all countries** to generate a complete set of weather requests.
- Ensures **uniform temporal coverage** across all datasets.

### 📊 Key Observations

- **Total Days**
  - 1461 days generated
  - Includes leap year (**2016**) → ensures accuracy in time-series analysis

- **Granularity**
  - Daily frequency (`freq="D"`)
  - Suitable for:
    - Historical weather analysis
    - Time-series modeling
    - Trend analysis

- **Continuity**
  - No missing dates in the sequence
  - Provides a **complete timeline backbone**

- **Scalability**
  - When combined with 164 countries:
    - **1461 × 164 = 239,604 total records**
  - Forms the foundation of a **fact table for weather data**

### ⚠️ Potential Issues

- **Data Volume Growth**
  - Cross join with countries will significantly increase dataset size
  - May impact:
    - Memory usage
    - API request volume

- **Unnecessary Dates**
  - Some dates may not have:
    - Available weather data
    - Business relevance (e.g., future extensions)


### ✅ Final Outcome

- A **complete and continuous date dimension** ready for:
  - Cross joining with countries 🌍
  - Generating full weather request coverage 🌦️
  - Supporting time-series analytics 📈

- Forms a **critical foundation for building a robust weather fact table** with consistent temporal granularity.

---


## Create Weather Request Matrix

The historical calendar is combined with the enterprise country dimension to
produce one weather request for every country and every date.

This request matrix defines the complete extraction scope for the weather ETL.

In [7]:
# ============================================================
# Create Weather Request Matrix
# ============================================================

weather_requests["key"] = 1
calendar["key"] = 1

weather_matrix = (
    weather_requests
    .merge(calendar, on="key")
    .drop(columns="key")
)

print("="*70)
print("WEATHER REQUEST MATRIX")
print("="*70)

print(f"Rows : {len(weather_matrix):,}")

display(weather_matrix.head())

WEATHER REQUEST MATRIX
Rows : 239,604


,iso2,country,city,date
0,AF,Afghanistan,Kabul,2015-01-01
1,AF,Afghanistan,Kabul,2015-01-02
2,AF,Afghanistan,Kabul,2015-01-03
3,AF,Afghanistan,Kabul,2015-01-04
4,AF,Afghanistan,Kabul,2015-01-05



---

### 🔍 Overview

- The **Weather Request Matrix (`weather_matrix`)** is created by performing a **cross join** between:
  - `weather_requests` (countries)
  - `calendar` (dates)
- This results in **one record per country per day**, defining the full scope for weather data extraction.

### 📊 Key Observations

- **Dataset Size**
  - Total rows: **239,604**
  - Derived from:
    - 164 countries × 1461 days

- **Structure**
  - Columns:
    - `iso2` → country code
    - `country` → country name
    - `city` → weather reference city
    - `date` → observation date

- **Complete Coverage**
  - Ensures:
    - No missing country-date combinations
    - Uniform time-series structure across all countries

- **ETL Readiness**
  - This table acts as the **driving table for API extraction**
  - Each row represents a **single weather API request**


### ⚠️ Potential Issues

- **Large Volume**
  - ~240K records → significant number of API calls
  - Risks:
    - API rate limits
    - Long execution time
    - Increased failure probability

- **Redundant Requests**
  - Some APIs allow:
    - Bulk date queries
    - Historical range queries
  - Row-by-row requests may be inefficient

- **Data Storage Impact**
  - Resulting weather dataset may grow large
  - Needs efficient storage (partitioning, compression)


### ✅ Final Outcome

- A **fully expanded request matrix** that:
  - Defines the complete weather extraction scope 🌦️
  - Ensures consistent country-date coverage 📅
  - Serves as the **foundation for scalable ETL execution** ⚙️

- Enables building a **robust weather fact table** aligned with enterprise data modeling standards.

---


## Retrieve Historical Weather

Historical weather observations are retrieved from the Open-Meteo Archive API.

Rather than requesting weather one day at a time, the complete historical
period is requested for each country's weather reference city in a single API
call.

This significantly reduces the number of API requests while ensuring complete
daily weather coverage for every country.

The extracted weather variables include:

- Mean Daily Temperature
- Daily Precipitation
- Maximum Wind Speed

The resulting dataset will later become the enterprise Weather Dimension.

---

### Step 1 — Prepare Country Coordinates

Before downloading weather, we need latitude and longitude for each weather reference city.

Since your dim_country already contains:

- `country`
- `city`

we'll build a coordinate table once using the Open-Meteo geocoding API.

In [8]:
# ============================================================
# Create Weather Coordinate Table
# ============================================================

coordinate_table = weather_requests.copy()

coordinate_table["latitude"] = None
coordinate_table["longitude"] = None

print("=" * 70)
print("WEATHER COORDINATE TABLE")
print("=" * 70)

print(f"Countries : {len(coordinate_table)}")

display(coordinate_table.head())

WEATHER COORDINATE TABLE
Countries : 164


,iso2,country,city,key,latitude,longitude
0,AF,Afghanistan,Kabul,1,None,None
1,AL,Albania,Tirana,1,None,None
2,DE,Germany,Berlin,1,None,None
3,AO,Angola,Luanda,1,None,None
4,SA,Saudi Arabia,Riyadh,1,None,None



---

### 🔍 Overview

- The **Weather Coordinate Table (`coordinate_table`)** is prepared to store:
  - `latitude`
  - `longitude`
- These coordinates are required to query the **Open-Meteo Archive API**, which operates on geographic coordinates rather than city names.
- The table is derived from `weather_requests`, ensuring consistency with the ETL pipeline.


### 📊 Key Observations

- **Dataset Size**
  - 164 rows → one per country
  - Matches the Weather Request Table

- **Structure**
  - Existing fields:
    - `iso2`, `country`, `city`
  - Newly added fields:
    - `latitude`, `longitude` (currently `None`)

- **One-Time Lookup**
  - Coordinates only need to be fetched **once per city**
  - Can be reused across all historical weather requests

- **Efficiency Gain**
  - Avoids repeated geocoding API calls during weather extraction
  - Improves performance and reduces API dependency


### ⚠️ Potential Issues

- **Missing Coordinates**
  - All values are initially `None`
  - Must be populated before weather extraction

- **City Ambiguity**
  - Some city names may not be unique globally
  - Risk of incorrect coordinates if not combined with country

- **API Dependency**
  - Geocoding relies on external API:
    - Possible rate limits
    - Network failures


### ✅ Final Outcome

- A **fully populated coordinate table** enabling:
  - Accurate weather API requests 📡
  - Efficient ETL execution ⚙️
  - Reusable geographic reference data 🌍

- Forms a **critical bridge between city-based data and coordinate-based weather APIs**.

---

### Step 2 — Geocoding Function

In [9]:
# ============================================================
# Geocoding Function
# ============================================================

def get_coordinates(city, country):

    url = (
        "https://geocoding-api.open-meteo.com/v1/search"
        f"?name={city}"
        f"&count=1"
        f"&language=en"
        f"&format=json"
    )

    response = requests.get(url)

    if response.status_code != 200:
        return None, None

    data = response.json()

    if "results" not in data:
        return None, None

    result = data["results"][0]

    return (
        result["latitude"],
        result["longitude"]
    )

### Step 3 — Retrieve Coordinates

In [10]:
# ============================================================
# Retrieve Coordinates
# ============================================================

for idx, row in coordinate_table.iterrows():

    lat, lon = get_coordinates(
        row["city"],
        row["country"]
    )

    coordinate_table.loc[idx, "latitude"] = lat
    coordinate_table.loc[idx, "longitude"] = lon

    print(
        f"{idx+1:03d}/164 : "
        f"{row['country']}"
    )

    time.sleep(0.25)

001/164 : Afghanistan
002/164 : Albania
003/164 : Germany
004/164 : Angola
005/164 : Saudi Arabia
006/164 : Algeria
007/164 : Argentina
008/164 : Armenia
009/164 : Australia
010/164 : Austria
011/164 : Azerbaijan
012/164 : Bangladesh
013/164 : Barbados
014/164 : Bahrain
015/164 : Belize
016/164 : Benin
017/164 : Belarus
018/164 : Bolivia
019/164 : Bosnia and Herzegovina
020/164 : Botswana
021/164 : Brazil
022/164 : Bulgaria
023/164 : Burkina Faso
024/164 : Burundi
025/164 : Bhutan
026/164 : Belgium
027/164 : Cambodia
028/164 : Cameroon
029/164 : Canada
030/164 : Chad
031/164 : Chile
032/164 : China
033/164 : Cyprus
034/164 : Colombia
035/164 : South Korea
036/164 : Costa Rica
037/164 : Côte d'Ivoire
038/164 : Croatia
039/164 : Cuba
040/164 : Denmark
041/164 : Ecuador
042/164 : Egypt
043/164 : El Salvador
044/164 : United Arab Emirates
045/164 : Eritrea
046/164 : Slovakia
047/164 : Slovenia
048/164 : Spain
049/164 : USA
050/164 : Estonia
051/164 : Ethiopia
052/164 : Philippines
053/164 


---

### 🔍 Overview

- The geocoding process retrieves **latitude and longitude** for each country’s reference city.
- A custom function `get_coordinates(city, country)` is used to query the geocoding API.
- The process iterates through all **164 countries**, populating the coordinate table.
- A small delay (`0.25 sec`) is added between requests to respect API rate limits.


### 📊 Key Observations

- **Full Coverage**
  - All 164 countries are processed sequentially
  - Each iteration updates:
    - `latitude`
    - `longitude`

- **Progress Tracking**
  - Real-time logging:
    ```
    001/164 : Afghanistan
    ...
    164/164 : Zimbabwe
    ```
  - Helps monitor execution and detect failures

- **Controlled API Usage**
  - `time.sleep(0.25)` reduces risk of:
    - Rate limiting
    - API blocking

- **Deterministic Execution**
  - Sequential loop ensures:
    - Traceability
    - Easier debugging


### ⚠️ Potential Issues

- **Slow Execution**
  - Sequential API calls → ~40+ seconds runtime
  - Not scalable for larger datasets

- **No Error Handling**
  - Failures (network/API issues) are not captured
  - Could leave missing coordinates unnoticed

- **Data Quality Risks**
  - Incorrect matches possible for:
    - Ambiguous city names
    - Special characters (e.g., Côte d'Ivoire)

- **Silent Failures**
  - No validation to confirm:
    - Coordinates were successfully retrieved


### 📈 Optimization Opportunities

| Area            | Current State        | Improvement Idea              |
|-----------------|---------------------|-------------------------------|
| Execution Speed | Sequential          | Parallel API calls            |
| Error Handling  | None                | Try/Except + Logging          |
| Data Validation | Not enforced        | Missing value checks          |
| Reusability     | In-memory only      | Persist to storage            |


### ✅ Final Outcome

- A **fully enriched coordinate table** with:
  - Accurate latitude & longitude for all countries 🌐
- Enables:
  - Efficient weather data extraction
  - Reduced API calls (one per country instead of per day)
- Forms a **critical foundation for the Weather ETL pipeline**

---

### 💡 Key Insight

> Performing geocoding once and reusing results is a **major optimization step**, reducing thousands of redundant API calls during historical weather extraction.


---

### Step 4 — Validate Coordinates

In [11]:
# ============================================================
# Coordinate Validation
# ============================================================

print("=" * 70)
print("WEATHER COORDINATE VALIDATION")
print("=" * 70)

print(f"Countries : {len(coordinate_table)}")

print("\nMissing Coordinates")

print(
    coordinate_table[
        ["latitude", "longitude"]
    ].isnull().sum()
)

print("\nSample")

display(coordinate_table.head())

WEATHER COORDINATE VALIDATION
Countries : 164

Missing Coordinates
latitude     2
longitude    2
dtype: int64

Sample


,iso2,country,city,key,latitude,longitude
0,AF,Afghanistan,Kabul,1,34.52813,69.17233
1,AL,Albania,Tirana,1,41.32744,19.81866
2,DE,Germany,Berlin,1,52.52437,13.41053
3,AO,Angola,Luanda,1,-8.83682,13.23432
4,SA,Saudi Arabia,Riyadh,1,24.68773,46.72185



---

### 🔍 Overview

- This step validates whether **latitude and longitude values** were successfully retrieved for all countries.
- Ensures readiness for the next phase: **historical weather extraction**.
- Includes:
  - Total country count
  - Missing value check
  - Sample preview of results


### 📊 Key Observations

- **Total Coverage**
  - 164 countries processed

- **Missing Values**
  - `latitude`: 2 missing
  - `longitude`: 2 missing
  - Indicates **2 countries failed geocoding**

- **Successful Retrieval**
  - Majority of records contain valid coordinates
  - Sample confirms:
    - Realistic geographic values
    - Proper mapping of cities to coordinates

- **Data Quality**
  - Coordinates appear consistent and usable for API calls
  - No obvious anomalies in sample output


### ⚠️ Potential Issues

- **Incomplete Coverage**
  - Even a small number of missing coordinates can:
    - Break downstream API calls
    - Cause incomplete weather data

- **Silent Failures**
  - Missing values may result from:
    - API lookup failure
    - Ambiguous city names
    - Temporary network issues

- **Pipeline Risk**
  - Weather extraction step depends **100% on valid coordinates**


### ✅ Final Outcome

- Coordinate dataset is **nearly complete and production-ready**
- Minor gaps identified and can be resolved quickly
- Ensures:
  - Reliable weather API requests 📡
  - Complete historical data coverage 📅


### 💡 Key Insight

> Even small data gaps at the **dimension level** can propagate into large-scale data loss in fact tables. Early validation ensures robustness of the entire ETL pipeline.

---


### Step 5 — Identify the Two Missing Countries

In [12]:
# ============================================================
# Countries with Missing Coordinates
# ============================================================

missing_coordinates = coordinate_table[
    coordinate_table["latitude"].isna()
].copy()

print("=" * 70)
print("COUNTRIES WITH MISSING COORDINATES")
print("=" * 70)

display(
    missing_coordinates[
        [
            "iso2",
            "country",
            "city"
        ]
    ]
)

COUNTRIES WITH MISSING COORDINATES


,iso2,country,city
79,KZ,Kazakhstan,Nur-Sultan
144,EH,Western Sahara,El-Aaiun



---

### 🔍 Overview

- This step isolates countries where **geocoding failed**.
- These records have **missing latitude and longitude values**.
- Identifying them early ensures **data completeness before weather extraction**.


### 📊 Key Observations

- **Total Missing Countries**: 2

| ISO2 | Country           | City        |
|------|------------------|-------------|
| KZ   | Kazakhstan       | Nur-Sultan  |
| EH   | Western Sahara   | El-Aaiun    |

- **Failure Rate**
  - 2 out of 164 → ~1.2%
  - Small but critical gap

- **Pattern Identified**
  - Issues likely due to:
    - **Recent city name changes**
      - *Nur-Sultan* (renamed back to *Astana*)
    - **Non-standard / disputed regions**
      - *Western Sahara*

### ⚠️ Root Cause Analysis

#### 1. Naming Inconsistencies
- APIs may not recognize:
  - Updated or temporary city names
  - Alternate spellings

#### 2. Geopolitical Sensitivity
- Some regions (e.g., Western Sahara) may:
  - Not be consistently indexed across APIs
  - Require alternative lookup strategies

#### 3. API Limitations
- Geocoding services may:
  - Return empty results
  - Fail silently for edge cases

### 📈 Impact Assessment

| Aspect              | Impact Level |
|--------------------|-------------|
| Data Completeness  | 🔴 High     |
| Pipeline Stability | 🔴 High     |
| Fix Effort         | 🟢 Low      |


### ✅ Final Outcome

- All missing coordinates can be **fully resolved with minimal effort**
- Ensures:
  - 100% country coverage 🌍
  - Reliable downstream weather extraction 📡
- Improves robustness of the **global weather ETL pipeline**

### 💡 Key Insight

> Edge cases in global datasets often arise from **naming inconsistencies and geopolitical variations**. Maintaining a small **manual override layer** dramatically improves data reliability.

---

## Manual Geocoding Corrections

A small number of locations cannot be resolved automatically through the
Open-Meteo geocoding service due to naming changes or geopolitical
differences.

These locations are corrected using standardized alternative place names.

Applying manual corrections ensures complete geographic coverage while keeping
the extraction process fully reproducible.

In [13]:
# ============================================================
# Manual Geocoding Corrections
# ============================================================

manual_city_mapping = {
    "Nur-Sultan": "Astana",
    "El-Aaiun": "Laayoune"
}

coordinate_table["geocoding_city"] = (
    coordinate_table["city"]
    .replace(manual_city_mapping)
)

In [14]:
# ============================================================
# Retry Missing Coordinates
# ============================================================

missing = coordinate_table[
    coordinate_table["latitude"].isna()
]

for idx, row in missing.iterrows():

    lat, lon = get_coordinates(
        row["geocoding_city"],
        row["country"]
    )

    coordinate_table.loc[idx, "latitude"] = lat
    coordinate_table.loc[idx, "longitude"] = lon

    print(
        f"{row['country']} -> "
        f"{row['geocoding_city']}"
    )

Kazakhstan -> Astana
Western Sahara -> Laayoune


In [15]:
# ============================================================
# Final Coordinate Validation
# ============================================================

print("=" * 70)
print("FINAL COORDINATE VALIDATION")
print("=" * 70)

print(
    coordinate_table[
        ["latitude", "longitude"]
    ].isnull().sum()
)

FINAL COORDINATE VALIDATION
latitude     0
longitude    0
dtype: int64


In [16]:
# ============================================================
# Export Weather Coordinate Table
# ============================================================

coordinate_table.to_csv(
    PROCESSED / "weather_coordinates.csv",
    index=False
)

print("=" * 70)
print("WEATHER COORDINATES EXPORTED")
print("=" * 70)

print(PROCESSED / "weather_coordinates.csv")

WEATHER COORDINATES EXPORTED
C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform\Datasets\Processed Datasets\weather_coordinates.csv



---

### 🔍 Overview

- A small number of locations failed initial geocoding due to:
  - City name changes
  - Alternative spellings
  - Geopolitical inconsistencies
- These cases were resolved using **standardized alternative city names** (`geocoding_city`).
- After correction, a **final validation** confirms complete coordinate coverage.


### 📊 Key Observations

- **Manual Corrections Applied**

| Country          | Original City | Corrected City |
|------------------|--------------|----------------|
| Kazakhstan       | Nur-Sultan   | Astana         |
| Western Sahara   | El-Aaiun     | Laayoune       |

- **Successful Retry**
  - All previously missing coordinates were successfully retrieved
  - Geocoding fallback strategy proved effective

- **Final Validation Results**
  - `latitude`: 0 missing ✅
  - `longitude`: 0 missing ✅
  - Dataset is now **100% complete**

### ✅ Data Quality Status

| Metric                  | Value |
|------------------------|------:|
| Total Countries        | 164   |
| Missing Coordinates    | 0     |
| Completion Rate        | 100%  |
| Data Readiness         | ✅ Production Ready |


### 🛠️ Best Practices Applied

- **Fallback Strategy**
  - Introduced alternate city names for failed lookups

- **Targeted Retry**
  - Only missing records were reprocessed → efficient & fast

- **Validation Gate**
  - Ensured zero missing values before proceeding

- **Reproducibility**
  - Corrections handled programmatically (not manual edits)


### 📈 Impact on ETL Pipeline

- Eliminates risk of:
  - Failed API calls
  - Missing weather data
- Ensures:
  - Full country coverage 🌍
  - Consistent historical data extraction 📅
  - Stable downstream transformations ⚙️


### 💡 Key Insight

> A small **manual correction layer** dramatically improves reliability in global datasets, where naming inconsistencies and geopolitical variations are unavoidable.

### 🚀 Final Outcome

- A **fully validated and complete coordinate table**
- Ready for:
  - High-volume weather data extraction
  - Enterprise-scale analytics use cases

- This step establishes a **robust geographic foundation** for the entire Weather ETL pipeline.

---

# Retrieve Historical Weather

The representative city coordinates generated in the previous section are now
used to retrieve historical daily weather observations from the Open-Meteo
Archive API.

Instead of collecting weather for thousands of ERP cities, the platform uses
one representative city per ERP country. This approach provides complete
country-level coverage while avoiding API limitations and sparse data.

The resulting dataset forms the Enterprise Weather Dimension and will later be
joined with procurement transactions using:

- Country
- Transaction Date

---


### Load Coordinate Table

In [17]:
# ============================================================
# Load Weather Coordinate Table
# ============================================================

weather_coordinates = pd.read_csv(
    PROCESSED / "weather_coordinates.csv"
)

print("=" * 70)
print("WEATHER COORDINATE TABLE")
print("=" * 70)

print(f"Countries : {weather_coordinates.shape[0]}")

display(weather_coordinates.head())

WEATHER COORDINATE TABLE
Countries : 164


,iso2,country,city,key,latitude,longitude,geocoding_city
0,AF,Afghanistan,Kabul,1,34.52813,69.17233,Kabul
1,AL,Albania,Tirana,1,41.32744,19.81866,Tirana
2,DE,Germany,Berlin,1,52.52437,13.41053,Berlin
3,AO,Angola,Luanda,1,-8.83682,13.23432,Luanda
4,SA,Saudi Arabia,Riyadh,1,24.68773,46.72185,Riyadh



---

### 🔍 Overview

- This step initiates the **weather data extraction phase** using the previously prepared coordinate table.
- Each country is represented by a **single reference city**, enabling:
  - Scalable API usage
  - Complete temporal coverage
  - Consistent country-level weather analysis

- Weather data will be retrieved from the **Open-Meteo Archive API** using:
  - `latitude`
  - `longitude`
  - Historical date range (2015–2018)


### 📊 Key Observations

- **Dataset Size**
  - 164 countries loaded successfully
  - Matches enterprise country dimension

- **Data Structure**
  - Core fields:
    - `iso2`, `country`, `city`
    - `latitude`, `longitude`
  - Additional field:
    - `geocoding_city` (used for fallback corrections)

- **Data Completeness**
  - 0 missing values for coordinates ✅
  - Fully validated in previous steps

- **Consistency**
  - One coordinate pair per country ensures:
    - Uniform granularity
    - Simplified joins with transactional data


### 🧠 Design Decision: Representative City Approach

Instead of:
- ❌ Thousands of ERP cities (high API cost, sparse data)

We use:
- ✅ One **representative city per country**

**Benefits:**
- Reduces API calls dramatically
- Avoids missing or inconsistent city-level weather data
- Ensures **complete daily coverage for all countries**


### 🔗 Downstream Integration

The resulting dataset (Weather Dimension) will be joined with procurement data using:

- `country`
- `transaction_date`

This enables:
- Weather-based demand analysis
- Supply chain risk modeling
- Seasonal procurement insights

### ⚠️ Potential Considerations

- **Granularity Trade-off**
  - Country-level weather may not capture regional variations

- **Urban Bias**
  - Capital/reference cities may not represent rural conditions

- **Time Alignment**
  - Ensure weather dates align exactly with transaction dates


### 📈 Data Readiness Summary

| Metric               | Value |
|---------------------|------:|
| Countries           | 164   |
| Missing Coordinates | 0     |
| Ready for API       | ✅ Yes |


### 🚀 Next Step

- Retrieve **historical daily weather data** for each country using the Open-Meteo API
- Build the **Enterprise Weather Dimension**

### 💡 Key Insight

> Using a **representative city per country** is a strategic trade-off that maximizes coverage and scalability while minimizing API complexity in large-scale data pipelines.

---

### Weather API Function

First create a reusable API function.

In [18]:
# ============================================================
# Weather API Function
# ============================================================

import requests
import time

def fetch_weather(latitude, longitude):

    url = (
        "https://archive-api.open-meteo.com/v1/archive"
    )

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": "2015-01-01",
        "end_date": "2018-12-31",
        "daily": [
            "temperature_2m_mean",
            "precipitation_sum",
            "rain_sum",
            "snowfall_sum",
            "wind_speed_10m_max"
        ],
        "timezone": "UTC"
    }

    response = requests.get(
        url,
        params=params,
        timeout=60
    )

    response.raise_for_status()

    return response.json()

### Test API

Always test before downloading 164 countries.

In [19]:
# ============================================================
# API Test
# ============================================================

sample = weather_coordinates.iloc[0]

result = fetch_weather(
    sample["latitude"],
    sample["longitude"]
)

print(result.keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])



---

## 📊 Observation

* The **API function (`fetch_weather`) is correctly structured**:

  * Uses the **Open-Meteo Archive API**
  * Pulls **daily historical weather data (2015–2018)**
  * Captures key meteorological variables:

    * Average temperature
    * Precipitation
    * Rain
    * Snowfall
    * Max wind speed

* **API test successful**:

  * Response contains expected keys:

    * `daily` → main dataset
    * `daily_units` → metadata
    * Location + timezone info


## 🔍 Insight

* **Reusable Design ✅**

  * Function is modular → can be applied across all 164 countries

* **Historical Consistency ✅**

  * Same date range as enterprise dataset → ensures:

    * Perfect temporal alignment
    * No data leakage

* **Rich Feature Potential 🚀**

  * Enables creation of:

    * Weather impact features
    * Seasonal indicators
    * Extreme condition flags

## ⚠️ Risk / Concern

* **API Rate Limits / Stability**

  * Looping over 164 countries may:

    * Trigger throttling
    * Cause intermittent failures

* **Response Size**

  * Multi-year daily data per country = **large payload**
  * Risk of:

    * Memory pressure
    * Slower ETL execution

* **Missing Data Edge Cases**

  * Some coordinates may return:

    * Partial or empty `daily` data

---


### Download Historical Weather

In [20]:
# ============================================================
# Download Historical Weather
# ============================================================

weather_data = []

failed_countries = []

total_countries = len(weather_coordinates)

print("=" * 70)
print("DOWNLOADING HISTORICAL WEATHER")
print("=" * 70)

for index, row in weather_coordinates.iterrows():
    try:
        result = fetch_weather(
            row["latitude"],
            row["longitude"]
        )

        daily = pd.DataFrame(result["daily"])

        daily["iso2"] = row["iso2"]
        daily["country"] = row["country"]
        daily["city"] = row["city"]

        weather_data.append(daily)

        print(
            f"[{index+1}/{total_countries}] "
            f"{row['country']} ✓"
        )

    except Exception as e:

        failed_countries.append({

            "iso2": row["iso2"],
            "country": row["country"],
            "error": str(e)

        })

        print(
            f"[{index+1}/{total_countries}] "
            f"{row['country']} ✗"
        )

    time.sleep(0.3)

DOWNLOADING HISTORICAL WEATHER
[1/164] Afghanistan ✓
[2/164] Albania ✓
[3/164] Germany ✓
[4/164] Angola ✓
[5/164] Saudi Arabia ✓
[6/164] Algeria ✓
[7/164] Argentina ✓
[8/164] Armenia ✓
[9/164] Australia ✓
[10/164] Austria ✓
[11/164] Azerbaijan ✓
[12/164] Bangladesh ✓
[13/164] Barbados ✓
[14/164] Bahrain ✓
[15/164] Belize ✓
[16/164] Benin ✓
[17/164] Belarus ✓
[18/164] Bolivia ✓
[19/164] Bosnia and Herzegovina ✓
[20/164] Botswana ✗
[21/164] Brazil ✗
[22/164] Bulgaria ✗
[23/164] Burkina Faso ✗
[24/164] Burundi ✗
[25/164] Bhutan ✗
[26/164] Belgium ✗
[27/164] Cambodia ✗
[28/164] Cameroon ✗
[29/164] Canada ✗
[30/164] Chad ✗
[31/164] Chile ✗
[32/164] China ✗
[33/164] Cyprus ✗
[34/164] Colombia ✗
[35/164] South Korea ✗
[36/164] Costa Rica ✗
[37/164] Côte d'Ivoire ✗
[38/164] Croatia ✗
[39/164] Cuba ✗
[40/164] Denmark ✓
[41/164] Ecuador ✓
[42/164] Egypt ✓
[43/164] El Salvador ✓
[44/164] United Arab Emirates ✓
[45/164] Eritrea ✓
[46/164] Slovakia ✓
[47/164] Slovenia ✓
[48/164] Spain ✓
[49/164] US


---

## 📊 Observation

* **Total Countries Processed:** 164

* **Successful Downloads:** ~70+ countries (partial success)

* **Failed Downloads:** ~90 countries (significant failure rate)

* **Pattern Identified:**

  * Early batch → Mostly ✅ success
  * Mid to later batch → Increasing ❌ failures
  * Intermittent recovery (some later successes)


## 🔍 Insight

* **This is NOT a data issue — it's an API execution issue**

The failure pattern clearly indicates:

* **API throttling / rate limiting**
* NOT invalid coordinates or bad countries

👉 Why?

* Countries like:

  * Canada, UK, France, China ❌ (should never fail logically)
* But later:

  * Sweden, Switzerland, South Africa ✅ (random recovery)

➡️ This inconsistency = **server-side request limiting**


## ⚠️ Root Cause Analysis

### 1. 🚫 Rate Limiting

* `time.sleep(0.3)` is **too aggressive**
* You're making:

  * ~3 requests per second
* Open-Meteo typically tolerates:

  * **~1 request per second (safe zone)**

### 2. ⏱️ No Retry Mechanism

* Any temporary failure = permanent loss
* No second attempt → high failure count

### 3. 🌐 Timeout / Network Instability

* Some failures may be:

  * Transient network issues
  * API latency spikes

---


### Combine Weather Data

In [21]:
# ============================================================
# Combine Weather Data
# ============================================================

dim_weather = pd.concat(
    weather_data,
    ignore_index=True
)

failed_weather = pd.DataFrame(
    failed_countries
)

print("=" * 70)
print("WEATHER DOWNLOAD SUMMARY")
print("=" * 70)

print(f"Successful Countries : {len(weather_data)}")
print(f"Failed Countries     : {len(failed_weather)}")
print(f"Rows                 : {dim_weather.shape[0]:,}")

if len(failed_weather) > 0:
    display(failed_weather)

WEATHER DOWNLOAD SUMMARY
Successful Countries : 75
Failed Countries     : 89
Rows                 : 109,575


,iso2,country,error
0,BW,Botswana,429 Client Error: Too Many Requests for url: h...
1,BR,Brazil,429 Client Error: Too Many Requests for url: h...
2,BG,Bulgaria,429 Client Error: Too Many Requests for url: h...
3,BF,Burkina Faso,429 Client Error: Too Many Requests for url: h...
4,BI,Burundi,429 Client Error: Too Many Requests for url: h...
...,...,...,...
84,VN,Vietnam,429 Client Error: Too Many Requests for url: h...
85,YE,Yemen,429 Client Error: Too Many Requests for url: h...
86,DJ,Djibouti,429 Client Error: Too Many Requests for url: h...
87,ZM,Zambia,429 Client Error: Too Many Requests for url: h...



---

## 📊 Observation

* **Total Countries Attempted:** 164

* **Successful Countries:** **75** ✅

* **Failed Countries:** **89** ❌

* **Total Rows Collected:** **109,575**

* **Failure Type:**

  * All failures show:

    > **429 Client Error: Too Many Requests**

## 🔍 Insight

* **Clear Confirmation of Rate Limiting**

  * 429 errors = API throttling (NOT data issues)
  * Your pipeline successfully fetched:

    * ~46% of global coverage
    * With **zero structural/data errors**

* **Data Integrity is Strong**

  * All successful records are:

    * Properly structured
    * Consistent across countries
    * Ready for downstream processing

* **Coverage Gap is Systematic**

  * Missing countries are **not random**
  * They are simply **unprocessed due to API limits**

## ⚠️ Critical Gap

* Current dataset:

  * ❌ Not globally complete
  * ❌ Biased toward early-request countries
  * ❌ Risky for production analytics (geographical skew)


## 📈 Expected Final Outcome

| Metric               | Current   | After Retry |
| -------------------- | --------- | ----------- |
| Country Coverage     | 75 / 164  | ✅ ~160+     |
| Completion %         | 46%       | ✅ ~98–100%  |
| Data Reliability     | Medium    | ✅ High      |
| Production Readiness | ❌ Partial | ✅ Full      |


### 🌍 End Result

You will have a **complete global weather intelligence layer**, enabling:

* Climate-aware procurement
* Logistics risk prediction
* Seasonality-driven forecasting

👉 This is a **major differentiator for SupplySight AI**

---


### Standardize Weather Dimension

In [22]:
# ============================================================
# Standardize Weather Dimension
# ============================================================

dim_weather.rename(
    columns={
        "time": "date",
        "temperature_2m_mean": "avg_temperature",
        "precipitation_sum": "precipitation",
        "rain_sum": "rainfall",
        "snowfall_sum": "snowfall",
        "wind_speed_10m_max": "max_wind_speed"
    },

    inplace=True
)

dim_weather["date"] = pd.to_datetime(
    dim_weather["date"]
)

### Reorder Columns

In [23]:
# ============================================================
# Reorder Columns
# ============================================================

dim_weather = dim_weather[
    [
        "date",
        "iso2",
        "country",
        "city",
        "avg_temperature",
        "precipitation",
        "rainfall",
        "snowfall",
        "max_wind_speed"
    ]
]

### Enterprise Validation

In [24]:
# ============================================================
# Enterprise Validation
# ============================================================

print("=" * 70)
print("ENTERPRISE WEATHER DIMENSION")
print("=" * 70)

print(f"Rows      : {dim_weather.shape[0]:,}")
print(f"Columns   : {dim_weather.shape[1]}")

print("\nCountries")

print(
    dim_weather["iso2"]
    .nunique()
)

print("\nDate Range")

print(

    dim_weather["date"].min(),
    "to",
    dim_weather["date"].max()
)

print("\nDuplicate Country-Date Keys")

print(
    dim_weather
    .duplicated(
        ["iso2", "date"]
    )
    .sum()
)

print("\nMissing Values")

print(
    dim_weather
    .isnull()
    .sum()
)

ENTERPRISE WEATHER DIMENSION
Rows      : 109,575
Columns   : 9

Countries
74

Date Range
2015-01-01 00:00:00 to 2018-12-31 00:00:00

Duplicate Country-Date Keys
0

Missing Values
date                  0
iso2               1461
country               0
city                  0
avg_temperature       0
precipitation         0
rainfall              0
snowfall              0
max_wind_speed        0
dtype: int64



---

## 📊 Validation Summary

* **Rows:** 109,575
* **Columns:** 9
* **Countries Covered:** 74 ⚠️ (should be ~160+)
* **Date Range:** ✅ 2015-01-01 → 2018-12-31 (complete & consistent)
* **Duplicate Keys (iso2 + date):** ✅ 0 (perfect)


## ⚠️ Critical Data Quality Issue

### ❌ Missing `iso2` Values: **1,461 rows**

This is a **serious integrity issue** because:

* `iso2` is your **primary join key**
* Without it:

  * ❌ Cannot join with enterprise dataset
  * ❌ Cannot map to country dimension
  * ❌ Breaks referential integrity


## 🔍 Root Cause

Most likely causes:

### 1. API Response Missing Mapping

* Weather API does NOT return ISO codes
* You are manually assigning it → some rows failed assignment

### 2. DataFrame Merge / Append Issue

During concatenation some batches may have:

* Missing `iso2` assignment
* Misaligned columns


## 📈 Final Data Quality Scorecard

| Check              | Status            |
| ------------------ | ----------------- |
| Schema Consistency | ✅                 |
| Date Coverage      | ✅                 |
| Duplicate Keys     | ✅                 |
| Missing Values     | ⚠️ (Fix required) |
| Country Coverage   | ⚠️ (Retry needed) |

---


In [25]:
print("=" * 70)
print("WEATHER DIMENSION VALIDATION")
print("=" * 70)

print("Rows :", len(dim_weather))
print("Countries :", dim_weather["iso2"].nunique())

print("\nDate Range")
print(
    dim_weather["date"].min(),
    "to",
    dim_weather["date"].max()
)

print("\nRows per Country (first 10)")
display(
    dim_weather.groupby("iso2")
               .size()
               .reset_index(name="days")
               .sort_values("iso2")
               .head(10)
)

WEATHER DIMENSION VALIDATION
Rows : 109575
Countries : 74

Date Range
2015-01-01 00:00:00 to 2018-12-31 00:00:00

Rows per Country (first 10)


,iso2,days
0,AE,1461
1,AF,1461
2,AL,1461
3,AM,1461
4,AO,1461
5,AR,1461
6,AT,1461
7,AU,1461
8,AZ,1461
9,BA,1461


In [26]:
print("=" * 70)
print("FAILED COUNTRIES")
print("=" * 70)

print(len(failed_weather))

if len(failed_weather) > 0:
    display(failed_weather.head(20))

FAILED COUNTRIES
89


,iso2,country,error
0,BW,Botswana,429 Client Error: Too Many Requests for url: h...
1,BR,Brazil,429 Client Error: Too Many Requests for url: h...
2,BG,Bulgaria,429 Client Error: Too Many Requests for url: h...
3,BF,Burkina Faso,429 Client Error: Too Many Requests for url: h...
4,BI,Burundi,429 Client Error: Too Many Requests for url: h...
5,BT,Bhutan,429 Client Error: Too Many Requests for url: h...
6,BE,Belgium,429 Client Error: Too Many Requests for url: h...
7,KH,Cambodia,429 Client Error: Too Many Requests for url: h...
8,CM,Cameroon,429 Client Error: Too Many Requests for url: h...
9,CA,Canada,429 Client Error: Too Many Requests for url: h...


### Create Retry Table

In [27]:
# ============================================================
# Retry Failed Countries
# ============================================================

retry_coordinates = weather_coordinates[
    weather_coordinates["iso2"].isin(
        failed_weather["iso2"]
    )
].reset_index(drop=True)

print("=" * 70)
print("COUNTRIES TO RETRY")
print("=" * 70)

print(f"Countries : {len(retry_coordinates)}")

display(retry_coordinates.head())

COUNTRIES TO RETRY
Countries : 89


,iso2,country,city,key,latitude,longitude,geocoding_city
0,BW,Botswana,Gaborone,1,-24.65451,25.90859,Gaborone
1,BR,Brazil,Brasilia,1,-15.77972,-47.92972,Brasilia
2,BG,Bulgaria,Sofia,1,42.69751,23.32415,Sofia
3,BF,Burkina Faso,Ouagadougou,1,12.36566,-1.53388,Ouagadougou
4,BI,Burundi,Gitega,1,-3.42708,29.92463,Gitega


### Retry Download (Improved)

This version:
- waits longer
- retries automatically
- handles HTTP 429 gracefully

In [28]:
# ============================================================
# Retry Weather Download
# ============================================================

import time

retry_weather = []

still_failed = []

for index, row in retry_coordinates.iterrows():
    success = False

    for attempt in range(5):
        try:
            result = fetch_weather(
                row["latitude"],
                row["longitude"]
            )

            daily = pd.DataFrame(result["daily"])

            daily["iso2"] = row["iso2"]
            daily["country"] = row["country"]
            daily["city"] = row["city"]

            retry_weather.append(daily)

            print(
                f"[{index+1}/{len(retry_coordinates)}] "
                f"{row['country']} ✓"
            )

            success = True

            break

        except Exception as e:
            print(
                f"{row['country']} "
                f"(Attempt {attempt+1})"
            )

            time.sleep(15)

    if not success:
        still_failed.append(row["iso2"])

    time.sleep(2)

[1/89] Botswana ✓
[2/89] Brazil ✓
[3/89] Bulgaria ✓
[4/89] Burkina Faso ✓
[5/89] Burundi ✓
[6/89] Bhutan ✓
[7/89] Belgium ✓
[8/89] Cambodia ✓
[9/89] Cameroon ✓
[10/89] Canada ✓
[11/89] Chad ✓
[12/89] Chile ✓
[13/89] China ✓
[14/89] Cyprus ✓
[15/89] Colombia ✓
[16/89] South Korea ✓
[17/89] Costa Rica ✓
Côte d'Ivoire (Attempt 1)
[18/89] Côte d'Ivoire ✓
[19/89] Croatia ✓
[20/89] Cuba ✓
[21/89] Philippines ✓
[22/89] Finland ✓
[23/89] France ✓
[24/89] Gabon ✓
[25/89] Georgia ✓
[26/89] Ghana ✓
[27/89] Greece ✓
[28/89] Guadeloupe ✓
[29/89] Guatemala ✓
French Guiana (Attempt 1)
[30/89] French Guiana ✓
[31/89] Guinea ✓
[32/89] Equatorial Guinea ✓
[33/89] Guinea-Bissau ✓
[34/89] Guyana ✓
[35/89] Haiti ✓
[36/89] Honduras ✓
[37/89] Hong Kong ✓
[38/89] Kenya ✓
[39/89] Kyrgyzstan ✓
[40/89] Kuwait ✓
[41/89] Laos ✓
Lesotho (Attempt 1)
[42/89] Lesotho ✓
[43/89] Liberia ✓
[44/89] Libya ✓
[45/89] Lithuania ✓
[46/89] Luxembourg ✓
[47/89] Lebanon ✓
[48/89] North Macedonia ✓
[49/89] Madagascar ✓
[50/89] Mal

### Append New Data

In [29]:
retry_weather = pd.concat(
    retry_weather,
    ignore_index=True
)

retry_weather.rename(
    columns={
        "time":"date",
        "temperature_2m_mean":"avg_temperature",
        "precipitation_sum":"precipitation",
        "rain_sum":"rainfall",
        "snowfall_sum":"snowfall",
        "wind_speed_10m_max":"max_wind_speed"
    },
    inplace=True
)

retry_weather["date"] = pd.to_datetime(
    retry_weather["date"]
)

retry_weather = retry_weather[
    [
        "date",
        "iso2",
        "country",
        "city",
        "avg_temperature",
        "precipitation",
        "rainfall",
        "snowfall",
        "max_wind_speed"
    ]
]

### Merge with Existing Weather

In [30]:
dim_weather = pd.concat(
    [
        dim_weather,
        retry_weather
    ],
    ignore_index=True
)

dim_weather.drop_duplicates(
    subset=[
        "iso2",
        "date"
    ],
    inplace=True
)

### Validate Again

In [31]:
print("=" * 70)
print("FINAL WEATHER VALIDATION")
print("=" * 70)

print("Rows :", len(dim_weather))
print("Countries :", dim_weather["iso2"].nunique())

print("\nStill Failed Countries")

print(len(still_failed))

FINAL WEATHER VALIDATION
Rows : 235221
Countries : 160

Still Failed Countries
3


### Check duplicate country-date keys

In [32]:
print("=" * 70)
print("DUPLICATE COUNTRY-DATE KEYS")
print("=" * 70)

duplicates = dim_weather.duplicated(
    subset=["iso2", "date"]
).sum()

print("Duplicate Keys :", duplicates)

DUPLICATE COUNTRY-DATE KEYS
Duplicate Keys : 0


### Identify the remaining failed countries

In [33]:
print("=" * 70)
print("REMAINING FAILED COUNTRIES")
print("=" * 70)

display(
    retry_coordinates[
        retry_coordinates["iso2"].isin(still_failed)
    ][["iso2", "country", "city"]]
)

REMAINING FAILED COUNTRIES


,iso2,country,city
81,UY,Uruguay,Montevideo
82,UZ,Uzbekistan,Tashkent
83,VE,Venezuela,Caracas


### Download only those 3 countries

Instead of rerunning everything, use this loop:

In [34]:
import time

for _, row in retry_coordinates[
    retry_coordinates["iso2"].isin(still_failed)
].iterrows():

    print(f"Downloading {row['country']}...")

    try:
        result = fetch_weather(
            row["latitude"],
            row["longitude"]
        )

        daily = pd.DataFrame(result["daily"])

        daily["iso2"] = row["iso2"]
        daily["country"] = row["country"]
        daily["city"] = row["city"]

        daily.rename(
            columns={
                "time": "date",
                "temperature_2m_mean": "avg_temperature",
                "precipitation_sum": "precipitation",
                "rain_sum": "rainfall",
                "snowfall_sum": "snowfall",
                "wind_speed_10m_max": "max_wind_speed"
            },
            inplace=True
        )

        daily["date"] = pd.to_datetime(daily["date"])

        daily = daily[
            [
                "date",
                "iso2",
                "country",
                "city",
                "avg_temperature",
                "precipitation",
                "rainfall",
                "snowfall",
                "max_wind_speed"
            ]
        ]

        dim_weather = pd.concat(
            [dim_weather, daily],
            ignore_index=True
        )

        dim_weather.drop_duplicates(
            subset=["iso2", "date"],
            inplace=True
        )

        print("✓ Success")

    except Exception as e:
        print(f"✗ Failed: {e}")

    time.sleep(20)

✓ Success
✓ Success
✓ Success


### Final validation

After downloading those 3 countries, rerun:

In [35]:
print("Rows :", len(dim_weather))
print("Countries :", dim_weather["iso2"].nunique())

print(
    "Duplicate Country-Date Keys :",
    dim_weather.duplicated(
        subset=["iso2", "date"]
    ).sum()
)

Rows : 239604
Countries : 163
Duplicate Country-Date Keys : 0


In [36]:
print("Unique ISO2 values:", dim_weather["iso2"].nunique())

print("\nRows by ISO2:")
counts = (
    dim_weather.groupby("iso2")
    .size()
    .reset_index(name="days")
    .sort_values("days")
)

display(counts.head(10))
display(counts.tail(10))

Unique ISO2 values: 163

Rows by ISO2:


,iso2,days
0,AE,1461
104,MX,1461
105,MY,1461
106,MZ,1461
107,NE,1461
108,NG,1461
109,NI,1461
110,NL,1461
103,MR,1461
111,NO,1461


,iso2,days
51,FR,1461
52,GA,1461
53,GB,1461
54,GE,1461
55,GF,1461
56,GH,1461
57,GM,1461
58,GN,1461
40,DK,1461
162,ZW,1461


In [37]:
print(dim_weather["iso2"].isnull().sum())
print((dim_weather["iso2"] == "").sum())

print(
    sorted(
        dim_weather["iso2"].astype(str).unique()
    )[-10:]
)

1461
0


TypeError: '<' not supported between instances of 'float' and 'str'

In [38]:
dim_weather[
    dim_weather["iso2"].isna()
][
    ["country", "city"]
].drop_duplicates()

,country,city
65745,Namibia,Windhoek


### Fix Namibia's ISO2

In [39]:
# ============================================================
# Fix Namibia ISO2 Code
# ============================================================

dim_weather.loc[
    dim_weather["country"] == "Namibia",
    "iso2"
] = "NA"

### Validate

In [40]:
print("=" * 70)
print("FINAL WEATHER VALIDATION")
print("=" * 70)

print("Rows :", len(dim_weather))
print("Countries :", dim_weather["iso2"].nunique())

print("Duplicate Country-Date Keys :")

print(
    dim_weather.duplicated(
        subset=["iso2", "date"]
    ).sum()
)

print("\nMissing Values")

print(
    dim_weather.isnull().sum()
)

FINAL WEATHER VALIDATION
Rows : 239604
Countries : 164
Duplicate Country-Date Keys :
0

Missing Values
date               0
iso2               0
country            0
city               0
avg_temperature    0
precipitation      0
rainfall           0
snowfall           0
max_wind_speed     0
dtype: int64


### Export

In [41]:
# ============================================================
# Export Enterprise Weather Dimension
# ============================================================

dim_weather.to_csv(
    PROCESSED / "dim_weather.csv",
    index=False
)

print("=" * 70)
print("ENTERPRISE WEATHER DIMENSION EXPORTED")
print("=" * 70)

print(PROCESSED / "dim_weather.csv")

ENTERPRISE WEATHER DIMENSION EXPORTED
C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform\Datasets\Processed Datasets\dim_weather.csv


In [45]:
check_weather = pd.read_csv(
    PROCESSED / "dim_weather.csv",
    dtype={"iso2": str},
    keep_default_na=False
)

print(check_weather.shape)
print(check_weather["iso2"].nunique())
print(check_weather["iso2"].isnull().sum())

(239604, 9)
164
0


# Notebook Summary

This notebook developed the **Enterprise Weather Dimension** for the SupplySight AI
Enterprise Procurement Decision Intelligence Platform.

Unlike traditional city-level weather integration, which often results in sparse
coverage and incomplete data, this notebook adopted a **country-level weather
strategy**. Each ERP country was represented by a standardized reference city,
whose geographic coordinates were used to retrieve historical daily weather
observations from the Open-Meteo Archive API.

The notebook first established a reliable geographic reference by standardizing
country names, assigning representative cities, and obtaining latitude and
longitude coordinates. These coordinates were then used to retrieve historical
daily weather observations covering the analysis period from **1 January 2015**
to **31 December 2018**.

The downloaded weather observations were validated, standardized, and transformed
into a complete Enterprise Weather Dimension that aligns directly with the ERP
transactions through a **Country + Date** integration strategy.

---

## Major Activities

- Loaded the Enterprise Country Dimension
- Standardized ERP country names
- Validated ISO country mappings
- Assigned representative reference cities
- Geocoded representative cities
- Applied manual geocoding corrections where required
- Generated the Enterprise Weather Coordinate Table
- Retrieved historical daily weather observations using the Open-Meteo Archive API
- Standardized weather variables and date formats
- Built the Enterprise Weather Dimension
- Validated data quality, completeness, and key uniqueness
- Exported enterprise-ready weather datasets

---

## Output Datasets

```
weather_coordinates.csv
```

Intermediate reference dataset containing standardized representative cities and
their geographic coordinates.

```
dim_weather.csv
```

Enterprise Weather Dimension containing daily historical weather observations
for all ERP countries.

---

## Final Dataset Statistics

| Metric | Value |
|--------|------:|
| Countries | 164 |
| Analysis Period | 2015–2018 |
| Calendar Days | 1,461 |
| Expected Records | 239,604 |
| Granularity | Country × Date |
| Duplicate Country-Date Keys | 0 |
| Missing Values | 0 |

---

## Business Value

The Enterprise Weather Dimension enriches procurement transactions with
historical environmental conditions that may influence supplier performance,
transportation efficiency, delivery delays, inventory planning, and procurement
risk.

Using a standardized Country-Date granularity provides significantly broader
coverage and greater consistency than attempting to collect weather data for
thousands of individual ERP cities, making the solution scalable and suitable
for enterprise analytics.

---

## Enterprise Integration Strategy

The Weather Dimension will be integrated with the procurement fact table using
the following composite business key:

```
Country (ISO2)
+
Transaction Date
```

This design aligns with the Enterprise Country Dimension and the Enterprise
Holiday Dimension, providing a consistent integration framework across all
external datasets.

---

## Next Step

The next notebook, **Notebook 01G – Country Holiday ETL**, builds the Enterprise
Holiday Dimension by generating historical public holiday calendars for all ERP
countries. Together, the Weather and Holiday Dimensions provide environmental
and business calendar intelligence that enrich procurement analytics and support
enterprise decision-making.